# 02 · Endpoint, réplicas, suelo y filtro de cohortes

El único endpoint es `intrinsic4`: LumA, LumB, HER2-enriched y Basal-like. No incluye tejido sano, Normal-like, HER2 clínico ni TNBC clínico.

In [ ]:
from pathlib import Path
import os, sys

DATA_PREPARE = Path.cwd().resolve()
while not (DATA_PREPARE / "combat_ref.py").exists():
    if DATA_PREPARE.parent == DATA_PREPARE:
        raise FileNotFoundError("Ejecuta este notebook dentro de data_prepare/.")
    DATA_PREPARE = DATA_PREPARE.parent
os.chdir(DATA_PREPARE)
if str(DATA_PREPARE) not in sys.path:
    sys.path.insert(0, str(DATA_PREPARE))
print("DATA_PREPARE:", DATA_PREPARE)

In [ ]:
import json
from pathlib import Path
import pandas as pd
from union_datasets import aplicar_endpoint, etiquetas_endpoint
from combat_ref import (
    Config, cargar, auditar_replicas, diagnosticar_suelo, corregir_suelo,
    filtrar_cohortes, filtro_deteccion, split_cohortes,
)

ENDPOINT_MODE = "intrinsic4"  # único endpoint permitido
ACTIVE_LABELS = etiquetas_endpoint(ENDPOINT_MODE)
cfg = Config(
    expr_path=str(DATA_PREPARE / "output_union/expr_raw_combined.csv"),
    meta_path=str(DATA_PREPARE / "output_union/metadata_combined.csv"),
    active_labels=ACTIVE_LABELS,
    min_labels_per_cohort=2,
    out_dir=str(DATA_PREPARE / "output_mcombat"),
)
print("Endpoint:", ENDPOINT_MODE, ACTIVE_LABELS)

In [ ]:
expr, meta = cargar(cfg)
expr, meta, _ = aplicar_endpoint(expr, meta, ENDPOINT_MODE)
expr, meta, audit = auditar_replicas(expr, meta, cfg)
diag = diagnosticar_suelo(expr, meta, cfg)
expr, correction = corregir_suelo(expr, meta, cfg, diag)
expr, meta, cohort_audit = filtrar_cohortes(expr, meta, cfg)
expr, genes_removed = filtro_deteccion(expr, meta, cfg)
split_table = split_cohortes(meta, cfg)
display(split_table)
display(pd.crosstab(meta[cfg.cohort_col], meta[cfg.label_col]))

In [ ]:
out = DATA_PREPARE / "output_prepared"
out.mkdir(parents=True, exist_ok=True)
expr.to_csv(out / "expr_prepared.csv")
meta.to_csv(out / "metadata_prepared.csv", index=False)
diag.to_csv(out / "diagnostico_suelo.csv")
cohort_audit.to_csv(out / "auditoria_cohortes.csv")
split_table.to_csv(out / "split_cohortes_preparado.csv")
if len(audit): audit.to_csv(out / "auditoria_replicas.csv", index=False)
if len(correction): correction.to_csv(out / "correccion_suelo.csv", index=False)
pd.Series(genes_removed, name="gene").to_csv(out / "genes_fuera_suelo.csv", index=False)
(out / "endpoint.json").write_text(json.dumps({
    "endpoint_mode": ENDPOINT_MODE,
    "active_labels": list(ACTIVE_LABELS),
    "min_labels_per_cohort": cfg.min_labels_per_cohort,
    "train_cohorts": list(cfg.train_cohorts_fixed),
    "test_cohorts": list(cfg.test_cohorts_fixed),
}, indent=2), encoding="utf-8")
print("Guardado:", out, expr.shape, meta.shape)